

"Cycle cutting"


is_eq is a top down version of what rebuild memoizes kind of.

is_eq(n) could retain a context of depth n of assumptions? That is sort of the rebuild repetition counter?
is_eq is following the congruence rules in a top down fashion


Extracting systems of equations



In [ ]:
from dataclasses import dataclass, field

type Id = int

@dataclass(frozen=True)
class Node:
    f : str
    args : tuple[Id, ...]

class EGraph:
    memo : dict[Node, Id] = field(default_factory=dict)
    uf : list[Id] = field(default_factory=list)

    def find(self, i : Id) -> Id:
        while self.uf[i] != i:
            i = self.uf[i]
        return i
    
    def union(self, i : Id, j : Id):
        i,j = self.find(i), self.find(j)
        if i != j:
            self.uf[i] = j
        return j

    def app(self, f, *args : Id) -> Id:
        args = tuple(self.find(a) for a in args)
        n = Node(f, args)
        if n in self.memo:
            return self.memo[n]
        else:
            i = len(self.uf)
            self.uf.append(i)
            self.memo[n] = i
            return i

    def is_eq_fast(self, i, j) -> bool:
        return self.find(i) == self.find(j)

    def nodes_in_class(self, i) -> list[Node]:
        return [n for n, c in self.memo.items() if self.find(c) == self.find(i)]

    def is_eq(self, i, j) -> bool:
        # This is kind of doing a multipattern Y = foo(bar(X)), Z = foo(bar(X)) -> z = y, but for all symbols  
        memo = {} # could retain between calls. is_eq_cache. uf _is_ is_eq_chache. Hmm.
        # seen = {}
        def worker(i,j):
            i,j = self.find(i), self.find(j)
            if i == j:
                memo[i,j] = True #
                return True
            if (i,j) in memo:
                return memo[i,j]
            memo[(i,j)] = False # Pessimisitically assume false. Or add a None
            for node in self.nodes_in_classes[i]:
                for node2 in self.nodes_in_classes[j]:
                    if node.f == node2.f and len(node.args) == len(node2.args):
                        if all(worker(a,b) for a,b in zip(node.args, node2.args)):
                            memo[(i,j)] = True
                            # self.union(i,j) ?
                            return True
        return worker(i,j)

    def rebuild(self):
        # I still have to rebuild nodes though?
        for i in range(len(self.uf)):
            for j in range(i+1, len(self.uf)):
                if self.is_eq(i,j):
                    self.union(i,j)




Priority queues. For horn clauses. Good rebuilding/compaction order?


In [ ]:
import heapq

class DecreaseKeyPQ:
    def __init__(self):
        self.heap = []
        self.current = {}  # item -> current priority

    def push(self, item, priority):
        self.current[item] = priority
        heapq.heappush(self.heap, (priority, item))

    def decrease(self, item, new_priority):
        # assume new_priority <= old
        self.current[item] = new_priority
        heapq.heappush(self.heap, (new_priority, item))

    def pop(self):
        while self.heap:
            priority, item = heapq.heappop(self.heap)
            if self.current.get(item) == priority:
                del self.current[item]
                return item, priority
        raise KeyError("pop from empty PQ")

    def __contains__(self, item):
        return item in self.current